## DATA 37711 HW 2
## Glen Morgenstern

### Problem 1(a)

The network $g:\mathbb{R}^p \to \mathbb{R}^2$ outputs two unconstrained real numbers,

$$
f_\mu(x) = g_1(x), \qquad f_\sigma(x) = \exp\big(g_2(x)\big)
$$

Reasoning why:
- $f_\mu(x) = \mathbb{E}[Y \mid X=x]$ can take any real valu, so no constraint is needed and the identify link is appropriate.
- $f_\sigma(x) = \sqrt{\mathrm{Var}(Y \mid X = x)}$ must be strictly positive. The exponential maps $\mathbb{R} to \to (0, \infty)$, so $g_2$ effectively models $\log \sigma$

### Problem 1(b)

Model. $Y \mid X = x \sim \mathcal{N}\big(f_\mu(x), f_\sigma(x)^2\big)$, so the density of one observation is

$$
p(y \mid x) = \frac{1}{f_\sigma(x)\sqrt{2\pi}} \exp\left( -\frac{(y - f_\mu(x))^2}{2 f_\sigma(x)^2} \right)
$$

Likelihood. Assuming the $n$ training pairs are independent,

$$
L(\theta) = \prod_{i=1}^{n} p(y_i \mid x_i)
$$

Log-likelihood. Taking the log turns the product into a sum:

$$
\log L(\theta) = \sum_{i=1}^{n} \left[ -\log f_\sigma(x_i) - \frac{1}{2}\log(2\pi) - \frac{(y_i - f_\mu(x_i))^2}{2 f_\sigma(x_i)^2} \right]
$$

Loss. Maximizing the likelihood is equivalent to minimizing the average negative log-likelihood:

$$
\mathcal{L}(\theta) = \frac{1}{n} \sum_{i=1}^{n} \left[ \log f_\sigma(x_i) + \frac{(y_i - f_\mu(x_i))^2}{2 f_\sigma(x_i)^2} \right] + \frac{1}{2}\log(2\pi)
$$

The constant $\frac{1}{2}\log(2\pi)$ does not depend on $\theta$ and can be dropped during optimization.


### Problem 1(c)

In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_parquet("taxi_trips.parquet")
X = df.drop(columns="trip_duration").values.astype("float32")
y = df["trip_duration"].values.astype("float32")

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=50)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=50)

def gauss_nll(y, mu, sigma):
    return np.mean(np.log(sigma) + 0.5*np.log(2*np.pi) + (y - mu)**2 / (2*sigma**2))

# Baseline 1: constant Gaussian
mu0, s0 = y_tr.mean(), y_tr.std()
print("const  val NLL:", gauss_nll(y_va, mu0, s0))

# Baseline 2: linear mean, constant sigma (sigma = std of training residuals)
lin = LinearRegression().fit(X_tr, y_tr)
s1 = np.std(y_tr - lin.predict(X_tr))
print("linear  val NLL:", gauss_nll(y_va, lin.predict(X_va), s1))

const  val NLL: 3.458691786832291
linear  val NLL: 3.4313552717486635


In [3]:
import torch, torch.nn as nn

class GaussianMLP(nn.Module):
    def __init__(self, p=7, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(p, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, 2) # g(x) in R^2, from part a
        )
    def forward(self, x):
        out = self.net(x)
        return out[:,0], out[:, 1] # mean, log-sigma
    
def nll_loss(mu, log_sigma, y):
    return (log_sigma + 0.5 * ((y-mu) / torch.exp(log_sigma))**2).mean()

Our current bar to beat is 3.431 (baseline with linear mean, constant sigma). The linear model only improved validation NLL by 0.027 vs. constant Gaussian.

In [ ]:
import torch, torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

#  standardize the target using TRAIN statistics only
ym, ys = y_tr.mean(), y_tr.std()
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=device)
Xtr_t, Xva_t, Xte_t = to_t(X_tr), to_t(X_va), to_t(X_te)
ytr_t, yva_t = to_t((y_tr - ym) / ys), to_t((y_va - ym) / ys)

# model: hetero=True learns sigma(x); hetero=False learns one global sigma 
class GaussianMLP(nn.Module):
    def __init__(self, p=7, h=128, hetero=True):
        super().__init__()
        self.hetero = hetero
        self.net = nn.Sequential(nn.Linear(p, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(),
                                 nn.Linear(h, 2))
        self.const_log_sigma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        out = self.net(x)
        mu = out[:, 0]
        log_sigma = out[:, 1] if self.hetero else self.const_log_sigma.expand_as(mu)
        return mu, log_sigma

def nll_loss(mu, log_sigma, y):
    return (log_sigma + 0.5 * ((y - mu) / torch.exp(log_sigma))**2).mean()

#  training loop with early stopping 
def train(model, epochs=100, lr=1e-3, bs=1024, patience=10):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, bad, n = float("inf"), None, 0, len(Xtr_t)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            mu, ls = model(Xtr_t[idx])
            loss = nll_loss(mu, ls, ytr_t[idx])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            val = nll_loss(*model(Xva_t), yva_t).item()
        print(f"epoch {epoch:3d}  val loss (standardized) {val:.4f}")
        if val < best - 1e-5:
            best, bad = val, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)   # roll back to the best epoch
    return model

#  convert predictions back to minutes so NLL is comparable to baselines 
def predict(model, X):
    model.eval()
    with torch.no_grad():
        mu, ls = model(X)
    mu, ls = mu.detach().cpu().numpy(), ls.detach().cpu().numpy()
    return mu * ys + ym, np.exp(ls) * ys

def report(name, model):
    mu, sigma = predict(model, Xva_t)
    print(f"{name}: val NLL {gauss_nll(y_va, mu, sigma):.4f}, "
          f"val RMSE {np.sqrt(np.mean((y_va - mu)**2)):.3f}")

torch.manual_seed(0); m_const  = train(GaussianMLP(hetero=False)); report("MLP const sigma ", m_const)
torch.manual_seed(0); m_hetero = train(GaussianMLP(hetero=True));  report("MLP hetero sigma", m_hetero)

We see the MLP beats the linear model by a lot. NLL drops from 3.43 to 2.72. The heteroscedastic model wins on NLL ob 0.067 but loses on RMSE to the constant sigma model.

In [ ]:
mu_t, sig_t = predict(m_hetero, Xte_t)
mc, sc = predict(m_const, Xte_t)
print("const Gaussian :", gauss_nll(y_te, mu0, s0))
print("linear         :", gauss_nll(y_te, lin.predict(X_te), s1))
print("MLP const sigma:", gauss_nll(y_te, mc, sc))
print("MLP hetero     :", gauss_nll(y_te, mu_t, sig_t))

On the test data, hetero loses by 0.076.

### Problem 1(d)

For a new input $x$, the network gives us $f_\mu(x)$ and $f_\sigma(x)$.

$$
Y \mid X = x \sim \mathcal{N}\big(f_\mu(x), f_\sigma(x)^2\big).
$$

The probability that $Y$ falls below 45 is the area under this density to the left of 45. Standardizing $Y$ gives

$$
P(Y < 45 \mid X = x) = P\left(\frac{Y - f_\mu(x)}{f_\sigma(x)} < \frac{45 - f_\mu(x)}{f_\sigma(x)}\right) = \Phi\left(\frac{45 - f_\mu(x)}{f_\sigma(x)}\right),
$$

The step works because $(Y - f_\mu(x)) / f_\sigma(x)$ follows a standard normal distribution when $Y \mid X = x$ is normal.
Importantly, the predicted probability depends on both $f_\mu$ and $f_\sigma$. Two trips with the same predicted mean can receive different probabilities if the model is more uncertain about one of them. A model with constant $\sigma$ cannot express this.



### Problem 1(e)

In [ ]:
from scipy.stats import norm

z_tr = (y_tr < 45).astype(int)   # 1 = under 45 min
z_te = (y_te < 45).astype(int)

def evaluate(name, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)          # avoid log(0)
    brier = np.mean((p - z_te)**2)
    ll    = -np.mean(z_te*np.log(p) + (1 - z_te)*np.log(1 - p))
    acc   = np.mean((p >= 0.5) == z_te)
    print(f"{name:24s} Brier {brier:.5f}  log-loss {ll:.5f}  acc {acc:.4f}")

n_te = len(y_te)
evaluate("always 'yes'",           np.ones(n_te))
evaluate("train base rate",        np.full(n_te, z_tr.mean()))
evaluate("linear, const sigma",    norm.cdf((45 - lin.predict(X_te)) / s1))
evaluate("MLP const sigma",        norm.cdf((45 - mc) / sc))
evaluate("MLP hetero sigma",       norm.cdf((45 - mu_t) / sig_t))

# Calibration sanity check: expected vs actual number of trips over 45 min
for name, p in {"MLP const": norm.cdf((45 - mc)/sc),
                "MLP hetero": norm.cdf((45 - mu_t)/sig_t)}.items():
    print(name, "expected # over 45:", round((1 - p).sum()), " actual:", (1 - z_te).sum())

- Accuracy is not informative here. Because about 99.55% of trips are under 45 minutes, a trivial classifier already achieves 99.55%. The MLPs improve this by only about 0.1 percentage points.
- The linear model is no better than the base rate on either proper scoring rule.
- Both MLPs improve Brier by roughly 30–35% over the base rate, with the constant-$\sigma$ model slightly ahead.
- Only the heteroscedastic model beats the base rate on log-loss (0.0227 vs 0.0288). The constant-$\sigma$ model is worse than the base rate on log-loss despite its better Brier score.

### Problem 2(a)

In [9]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("retail.csv")
for c in ["StockCode", "CustomerID"]:
    df[c] = df[c].astype(str).astype("category")   # str first so 17850.0 and "85123A" are both plain labels

for c in ["StockCode", "CustomerID"]:
    print(c, "unique:", df[c].nunique())
print("one-hot width:", df["StockCode"].nunique() + df["CustomerID"].nunique())

tr, tmp = train_test_split(df, test_size=0.30, random_state=0)
va, te = train_test_split(tmp, test_size=0.50, random_state=0)
print(len(tr), len(va), len(te))

# how many val/test rows have a category never seen in training?
for c in ["StockCode", "CustomerID"]:
    seen = set(tr[c].astype(str))
    for name, part in [("val", va), ("test", te)]:
        unseen = (~part[c].astype(str).isin(seen)).mean()
        print(f"{c} unseen in {name}: {unseen:.2%} of rows")

StockCode unique: 3665
CustomerID unique: 4339
one-hot width: 8004
278546 59689 59689
StockCode unseen in val: 0.07% of rows
StockCode unseen in test: 0.05% of rows
CustomerID unseen in val: 0.04% of rows
CustomerID unseen in test: 0.04% of rows


Cardinality. StockCode has 3,665 unique values and CustomerID has 4,339, across 397,924 transactions. Both are high-cardinality with a long tail. The median category has about 40-50 rows, but about 15% of products and 7% of customers have five or fewer, and 4.6% of prodcuts and 1.7% of customers appear exactly once.

Is one-hot encoding feasible? It would produce 3,665 + 4,339 = 8,004 columns. That is a huge matrix, which is impractical. A sparse representation could fit, but it remains a poor representation. The model has a learn a separate weight for each of more than 8,000 columns, and estimate it from very few rows for the rare categories. So we would want compact encodings: taret encoding (one number per category) and learned embeddings (a short vector per category).


### Problem 2(b)

In [ ]:
import category_encoders as ce
import torch, torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

cols = ["StockCode", "CustomerID"]
ytr, yva, yte = [np.log(d["Quantity"].values).astype("float32") for d in (tr, va, te)]

#  target encoding: fit on TRAIN only, apply to all splits 
enc = ce.TargetEncoder(cols=cols, min_samples_leaf=20, smoothing=10)
enc.fit(tr[cols].astype(str), ytr)
Xtr, Xva, Xte = [enc.transform(d[cols].astype(str)).values.astype("float32")
                 for d in (tr, va, te)]

#  baselines (validation RMSE on the log scale) 
rmse = lambda a, b: np.sqrt(np.mean((a - b)**2))
print("global mean         :", rmse(yva, ytr.mean()))
print("StockCode encoding  :", rmse(yva, Xva[:, 0]))
print("CustomerID encoding :", rmse(yva, Xva[:, 1]))

#  MLP on the two encoded features 
class MLP(nn.Module):
    def __init__(self, p=2, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(p, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(),
                                 nn.Linear(h, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def fit_mlp(model, Xtr, ytr, Xva, yva, epochs=100, lr=1e-3, bs=1024, patience=10):
    Xtr, ytr, Xva, yva = [torch.tensor(a, device=device) for a in (Xtr, ytr, Xva, yva)]
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            loss = ((model(Xtr[idx]) - ytr[idx])**2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val = ((model(Xva) - yva)**2).mean().item()
        if val < best - 1e-5:
            best, bad = val, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    print(f"stopped after {epoch + 1} epochs, best val MSE {best:.4f}")
    model.load_state_dict(best_state)
    return model

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, device=device)).cpu().numpy()

m_te = fit_mlp(MLP(), Xtr, ytr, Xva, yva)
print("target-enc MLP val RMSE (log):", rmse(yva, predict(m_te, Xva)))

The combination of target encodings on both StockCode and CustomerID beats either alone by a fairly wide margin (0.881 to 0.709). So the model is using both features together, not just picking the better one.